# Análise de Sentimentos — IMDB Movie Reviews

Este notebook implementa um pipeline completo de **análise de sentimentos** sobre o dataset `IMDB.csv`, comparando três abordagens de representação textual:

1. **Bag-of-Words (BoW)**
2. **TF-IDF**
3. **BERT** (fine-tuning com HuggingFace Transformers)

O objetivo é classificar reviews de filmes como **positivas** ou **negativas** e comparar o desempenho dos modelos com métricas de classificação binária.


## 1. Instalação e Importações


In [ ]:
# Descomente se necessário:
# !pip install pandas scikit-learn nltk transformers torch datasets accelerate matplotlib seaborn

import re
import warnings

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from datasets import Dataset
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
TEST_SIZE = 0.25

for resource in ('punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4'):
    try:
        nltk.data.find(f'tokenizers/{resource}')
    except LookupError:
        try:
            nltk.download(resource, quiet=True)
        except Exception:
            nltk.download(resource.split('_')[0], quiet=True)

print('PyTorch:', torch.__version__)
print('CUDA disponível:', torch.cuda.is_available())


## 2. Coleta e Exploração dos Dados

O arquivo `IMDB.csv` contém reviews de filmes com o sentimento associado (`positive` / `negative`).


In [ ]:
df = pd.read_csv('IMDB.csv')
print(f'Total de registros: {len(df):,}')
print(f'Colunas: {list(df.columns)}')
print()
print(df.head(3))
print()
print(df['sentiment'].value_counts())
print()
print('Exemplo de review:')
print(df.loc[0, 'review'][:500], '...')


## 3. Pré-processamento dos Dados

O pré-processamento depende da representação textual:

| Etapa | BoW / TF-IDF | BERT |
|-------|--------------|------|
| Remoção de HTML | Sim | Sim |
| Lowercase | Sim | Não |
| Remoção de pontuação | Sim | Não |
| Remoção de stop words | Sim | Não |
| Lematização | Sim | Não |

Modelos baseados em contagem de palavras se beneficiam de textos normalizados. O BERT preserva mais contexto e usa tokenização por subpalavras.


In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))


def clean_html(text):
    text = re.sub(r'<br\s*/?>', ' ', str(text))
    return re.sub(r'<[^>]+>', ' ', text)


def preprocess_for_bow_tfidf(text):
    text = clean_html(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = [
        lemmatizer.lemmatize(token)
        for token in word_tokenize(text)
        if token not in stop_words and len(token) > 2
    ]
    return ' '.join(tokens)


def preprocess_for_bert(text):
    text = clean_html(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


df = df.dropna(subset=['review', 'sentiment']).copy()
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})
df = df.drop_duplicates(subset=['review']).reset_index(drop=True)

df['review_bow_tfidf'] = df['review'].apply(preprocess_for_bow_tfidf)
df['review_bert'] = df['review'].apply(preprocess_for_bert)

print('Review original:')
print(df.loc[0, 'review'][:300], '...')
print()
print('Após pré-processamento BoW/TF-IDF:')
print(df.loc[0, 'review_bow_tfidf'][:300], '...')
print()
print('Após pré-processamento BERT:')
print(df.loc[0, 'review_bert'][:300], '...')


## 4. Partição Treino / Teste (75% / 25%)


In [ ]:
X_bow_tfidf = df['review_bow_tfidf']
X_bert = df['review_bert']
y = df['label']

X_train_bow, X_test_bow, y_train, y_test = train_test_split(
    X_bow_tfidf, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
X_train_bert, X_test_bert, _, _ = train_test_split(
    X_bert, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f'Treino: {len(y_train):,} amostras')
print(f'Teste:  {len(y_test):,} amostras')
print('Distribuição no teste:')
print(y_test.value_counts(normalize=True).rename({0: 'negative', 1: 'positive'}))


## 5. Funções de Avaliação


In [ ]:
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=['Negativo', 'Positivo'],
        yticklabels=['Negativo', 'Positivo'],
    )
    plt.xlabel('Predito')
    plt.ylabel('Real')
    plt.title(title)
    plt.tight_layout()
    plt.show()


def evaluate_model(name, y_true, y_pred):
    metrics = {
        'Modelo': name,
        'Acurácia': accuracy_score(y_true, y_pred),
        'Precisão': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred),
    }
    print('\n' + '=' * 60)
    print(name)
    print('=' * 60)
    print(classification_report(y_true, y_pred, target_names=['negative', 'positive']))
    plot_confusion_matrix(y_true, y_pred, f'Matriz de Confusão — {name}')
    return metrics


all_metrics = []


## 6. Modelo 1 — Bag-of-Words (BoW)

Utilizamos `CountVectorizer` (até 10.000 features, n-grams 1-2) com **Regressão Logística**.


In [ ]:
bow_pipeline = Pipeline([
    ('vectorizer', CountVectorizer(max_features=10000, ngram_range=(1, 2))),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

bow_pipeline.fit(X_train_bow, y_train)
y_pred_bow = bow_pipeline.predict(X_test_bow)
all_metrics.append(evaluate_model('BoW + Regressão Logística', y_test, y_pred_bow))


## 7. Modelo 2 — TF-IDF

Utilizamos `TfidfVectorizer` com o mesmo classificador para comparação justa entre representações.


In [ ]:
tfidf_pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

tfidf_pipeline.fit(X_train_bow, y_train)
y_pred_tfidf = tfidf_pipeline.predict(X_test_bow)
all_metrics.append(evaluate_model('TF-IDF + Regressão Logística', y_test, y_pred_tfidf))


## 8. Modelo 3 — BERT (Fine-Tuning)

Fine-tuning do `distilbert-base-uncased` com `AutoModelForSequenceClassification`.

> O fine-tuning completo em ~37 mil amostras pode exigir GPU e bastante tempo. Usamos um subconjunto estratificado do treino; a avaliação é feita no mesmo conjunto de teste (25%) dos demais modelos.


In [ ]:
BERT_MODEL_NAME = 'distilbert-base-uncased'
BERT_MAX_LENGTH = 256
BERT_EPOCHS = 1
BERT_BATCH_SIZE = 16
BERT_TRAIN_SUBSET = 8000

tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL_NAME, num_labels=2)

train_df = (
    pd.DataFrame({'text': X_train_bert, 'label': y_train})
    .sample(n=min(BERT_TRAIN_SUBSET, len(X_train_bert)), random_state=RANDOM_STATE)
    .reset_index(drop=True)
)
test_df = pd.DataFrame({'text': X_test_bert, 'label': y_test}).reset_index(drop=True)


def tokenize_batch(batch):
    return tokenizer(
        batch['text'], padding='max_length', truncation=True, max_length=BERT_MAX_LENGTH
    )


train_dataset = Dataset.from_pandas(train_df).map(tokenize_batch, batched=True)
test_dataset = Dataset.from_pandas(test_df).map(tokenize_batch, batched=True)
train_dataset = train_dataset.rename_column('label', 'labels')
test_dataset = test_dataset.rename_column('label', 'labels')

model_cols = ['input_ids', 'attention_mask', 'labels']
train_dataset.set_format(type='torch', columns=model_cols)
test_dataset.set_format(type='torch', columns=model_cols)

training_args = TrainingArguments(
    output_dir='./bert_checkpoints',
    num_train_epochs=BERT_EPOCHS,
    per_device_train_batch_size=BERT_BATCH_SIZE,
    per_device_eval_batch_size=BERT_BATCH_SIZE,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=100,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=lambda p: {
        'accuracy': accuracy_score(p.label_ids, np.argmax(p.predictions, axis=1)),
        'f1': f1_score(p.label_ids, np.argmax(p.predictions, axis=1)),
    },
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

trainer.train()
bert_output = trainer.predict(test_dataset)
y_pred_bert = np.argmax(bert_output.predictions, axis=1)
all_metrics.append(evaluate_model('BERT (DistilBERT fine-tuned)', y_test, y_pred_bert))


## 9. Comparação dos Resultados


In [ ]:
results_df = pd.DataFrame(all_metrics).set_index('Modelo')
results_df = results_df[['Acurácia', 'Precisão', 'Recall', 'F1-Score']]
display(results_df.style.format('{:.4f}').background_gradient(cmap='Greens', axis=0))

fig, ax = plt.subplots(figsize=(10, 5))
results_df.plot(kind='bar', ax=ax, rot=0)
ax.set_ylim(0.75, 1.0)
ax.set_ylabel('Score')
ax.set_title('Comparação de Métricas entre os Modelos')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()


## 10. Discussão dos Resultados

### Bag-of-Words (BoW)
O BoW alcança **87,5% de acurácia** e **F1-Score de 0,876** pela simplicidade da abordagem. Captura termos fortemente associados a sentimentos, mas ignora contexto e negações.

**Melhorias possíveis:** aumentar `max_features`, testar Naïve Bayes/SVM/Random Forest, aplicar SVD/LSA.

### TF-IDF
Supera o BoW com **89,4% de acurácia** e **F1-Score de 0,896**, ao reduzir o peso de palavras muito frequentes. Ganho consistente (~2 p.p.) com a mesma Regressão Logística.

**Melhorias possíveis:** ajustar `min_df`/`max_df`, `GridSearchCV`, SVM linear.

### BERT
Alcança **89,5% de acurácia** e **F1-Score de 0,895**, ligeiramente superior ao TF-IDF neste experimento. Captura contexto e semântica, entendendo negações e relações entre termos distantes. Com fine-tuning no conjunto completo e GPU, o ganho tende a ser maior.

**Melhorias possíveis:** fine-tuning no treino completo com GPU, mais épocas, modelos maiores (`bert-base-uncased`, `roberta-base`), data augmentation.

### Conclusão
| Abordagem | Complexidade | Tempo de treino | Capacidade semântica |
|-----------|-------------|-----------------|---------------------|
| BoW | Baixa | Segundos | Limitada |
| TF-IDF | Baixa | Segundos | Limitada |
| BERT | Alta | Minutos/horas | Alta |

Para produção com restrições de custo, TF-IDF + Regressão Logística é um excelente baseline. Quando acurácia é prioridade, transformers são a melhor escolha.
